# 05 — Filtragem colaborativa usuário–usuário

**DATA SCIENCE KNOWLEDGE BASE**  
**Área:** `09_recommender_systems` — Sistemas de Recomendação  
**Capítulo:** 05 de 16  

> Execute as células em sequência, de cima para baixo, em um kernel reiniciado. Use o **Guia Técnico** e a **Cheat Sheet** da seção Materiais complementares como referências para este capítulo.

## Objetivos de aprendizagem
- Construir matriz usuário–item
- Calcular similaridade com usuários que compartilham avaliações

## Materiais complementares

- [Guia Técnico](reference_materials/05_similaridade_usuarios.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_05_similaridade_usuarios.pdf)

**Navegação:** os links são relativos à pasta deste notebook. A versão editorial desses PDFs será revalidada após a consolidação científica dos notebooks.

## Fundamentação conceitual
Métodos user–user procuram usuários com comportamento semelhante. O cosseno nos vetores com zeros de ausência é apenas um **baseline operacional**; zeros não são avaliações, e a similaridade pode sofrer viés de popularidade.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## Matriz explícita

In [2]:
subset=ratings[ratings.movieId.isin(ratings.movieId.value_counts().head(300).index)]
mat=subset.pivot_table(index='userId',columns='movieId',values='rating')
print(mat.shape,'observado:',int(mat.notna().sum().sum()))

(603, 300) observado: 32927


**Interpretação.** Mantenha NaN como indicação de avaliação ausente.

## Vizinhos aproximados

In [3]:
filled=mat.fillna(0)
S=cosine_similarity(filled)
u=1; idx=mat.index.get_loc(u)
order=np.argsort(-S[idx]); neighbors=[(int(mat.index[j]),round(float(S[idx,j]),3)) for j in order if j!=idx][:5]
print('Usuários semelhantes:',neighbors)

Usuários semelhantes: [(266, 0.621), (469, 0.61), (91, 0.601), (599, 0.598), (135, 0.588)]


**Interpretação.** Não compare a diagonal: cada usuário é idêntico a si mesmo.

## Candidatos ainda não vistos

In [4]:
neighbor=neighbors[0][0]; unseen=mat.loc[neighbor].dropna().index.difference(mat.loc[u].dropna().index)
print('Itens sugeridos pelo vizinho:',list(unseen[:10]))

Itens sugeridos pelo vizinho: [16, 17, 21, 32, 39, 95, 288, 353, 368, 377]


**Interpretação.** A etapa de candidatos é diferente da avaliação de relevância.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** vizinhos e coavaliações. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **vizinhos e coavaliações** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **vizinhos e coavaliações** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — Similaridade apenas nos itens coavaliados

O cosseno de vetores com `fillna(0)` é uma aproximação computacional, mas **zero representa ausência** e não uma nota. Para interpretar um vizinho, verifique o número de itens coavaliados e considere centralização, regularização de suporte e um mínimo de observações.

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [5]:
# Comparação interpretável para vizinhos: suporte de coavaliações
suportes = []
for vizinho, similaridade in neighbors:
    n_co = int((mat.loc[u].notna() & mat.loc[vizinho].notna()).sum())
    suportes.append((vizinho, round(float(similaridade),3), n_co))
print('Vizinho, cosseno com zeros, quantidade coavaliada:', suportes)
assert all(n >= 0 for _,_,n in suportes)

Vizinho, cosseno com zeros, quantidade coavaliada: [(266, 0.621, 52), (469, 0.61, 68), (91, 0.601, 81), (599, 0.598, 86), (135, 0.588, 57)]


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

## Limitações e boas práticas
Zeros imputados são conveniência computacional; uma alternativa é similaridade somente sobre itens coavaliados e encolhimento por suporte.

## Exercícios
Compare a vizinhança para dois usuários e explique a sensibilidade à esparsidade.

## Soluções comentadas
Troque `u` por outro ID presente em `mat.index` e recalcule a linha de `S`.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados do capítulo em `outputs/`.


In [6]:
# EP_OUTPUTS_09_V1
# Exportação após os experimentos. Executar este notebook desde o início (Run All).
# Arquivos com prefixo do capítulo evitam colisões com outras saídas.
from pathlib import Path
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame(suportes, columns=["usuario_vizinho", "similaridade_cosseno_com_ausencias_zero",
                                 "itens_coavaliados"]).assign(usuario_referencia=u).to_csv(
    out / "05_vizinhos_usuarios.csv", index=False)
print('Outputs do capítulo 05 salvos em:', out.resolve())


Outputs do capítulo 05 salvos em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/09_recommender_systems/outputs
